In [1]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
OUT_ROOT = Path(r"D:\SemanticBiods\data")
LANGUAGES = ["English", "French"]
RUNS = ["main", "content", "window"]
OUTCOMES = ["RSC", "Turnover"]
SPECS = {"main": [0, 1, 2, 3, 4, 5], "lag1plus": [1, 2, 3, 4, 5]}
CURVES = {"PastChange": ("M2_History", ["Frequency", "PastChange"]),
          "Global": ("M3_Global", ["Frequency", "PastChange", "Global"]),
          "Local": ("M4_Local", ["Frequency", "PastChange", "Local"])}
CONTRASTS = [
    ("History_vs_Frequency", "M1_Frequency", "M2_History"),
    ("Global_vs_History", "M2_History", "M3_Global"),
    ("Local_vs_History", "M2_History", "M4_Local"),
    ("Local_vs_Global", "M3_Global", "M4_Local"),
    ("Full_vs_History", "M2_History", "M5_Full"),
    ("Full_vs_Local", "M4_Local", "M5_Full"),
]
N_BOOT = 2000
SEED = 20260901
STEP = 10


def demean_within(values, keys):
    s = pd.Series(np.asarray(values, dtype=float))
    return (s - s.groupby(pd.Series(np.asarray(keys))).transform("mean")).to_numpy()


def prepare(df, lags):
    cols = [f"{f}_lag{l}" for f in ["Frequency", "Global", "Local",
                                    "PastChange"] for l in lags]
    X = df[cols].to_numpy(float)
    y = df["Outcome"].to_numpy(float)
    return X, (y - y.mean()) / y.std(), cols


def word_slices(codes):
    order = np.argsort(codes, kind="stable")
    cuts = np.r_[0, np.flatnonzero(np.diff(codes[order])) + 1, len(order)]
    return [order[lo:hi] for lo, hi in zip(cuts[:-1], cuts[1:])]


def sufficient_stats(X, y, codes, n_words):
    p = X.shape[1]
    st = {"n": np.zeros(n_words), "sx": np.zeros((n_words, p)),
          "sx2": np.zeros((n_words, p)), "sy": np.zeros(n_words),
          "xty": np.zeros((n_words, p)), "xtx": np.zeros((n_words, p, p))}
    for idx in word_slices(codes):
        g = codes[idx[0]]
        Xi, yi = X[idx], y[idx]
        st["n"][g] = len(idx)
        st["sx"][g] = Xi.sum(0)
        st["sx2"][g] = (Xi ** 2).sum(0)
        st["sy"][g] = yi.sum()
        st["xty"][g] = Xi.T @ yi
        st["xtx"][g] = Xi.T @ Xi
    return st


def ridge_from_stats(N, sx, sx2, sy, xty, xtx, alpha):
    """Exactly standardise-then-ridge-with-intercept, from moments only."""
    mean = sx / N
    sd = np.sqrt(np.maximum(sx2 / N - mean ** 2, 1e-15))
    ztz = (xtx - N * np.outer(mean, mean)) / np.outer(sd, sd)
    zty = (xty - mean * sy) / sd
    return np.linalg.solve(ztz + alpha * np.eye(len(sx)), zty)


def bootstrap_coefficients(X, y, codes, n_words, alpha, rng, batch=100):
    st = sufficient_stats(X, y, codes, n_words)
    flat = st["xtx"].reshape(n_words, -1)
    p = X.shape[1]
    out = np.empty((N_BOOT, p))

    done = 0
    while done < N_BOOT:
        m = min(batch, N_BOOT - done)
        counts = np.zeros((m, n_words))
        for i in range(m):
            counts[i] = np.bincount(rng.integers(0, n_words, n_words),
                                    minlength=n_words)
        N, sx = counts @ st["n"], counts @ st["sx"]
        sx2, sy = counts @ st["sx2"], counts @ st["sy"]
        xty = counts @ st["xty"]
        xtx = (counts @ flat).reshape(m, p, p)
        for i in range(m):
            out[done + i] = ridge_from_stats(N[i], sx[i], sx2[i], sy[i],
                                             xty[i], xtx[i], alpha)
        done += m
    return out


def bootstrap_r2(observed, predictions, codes, n_words, rng, batch=200):
    names = list(predictions)
    slices = word_slices(codes)
    sse = np.column_stack(
        [[float(((observed[s] - predictions[m][s]) ** 2).sum())
          for s in slices] for m in names])
    sy = np.array([observed[s].sum() for s in slices])
    sy2 = np.array([(observed[s] ** 2).sum() for s in slices])
    nn = np.array([len(s) for s in slices], dtype=float)

    draws = np.empty((N_BOOT, len(names)))
    done = 0
    while done < N_BOOT:
        m = min(batch, N_BOOT - done)
        counts = np.zeros((m, n_words))
        for i in range(m):
            counts[i] = np.bincount(rng.integers(0, n_words, n_words),
                                    minlength=n_words)
        N = counts @ nn
        sst = (counts @ sy2) - (counts @ sy) ** 2 / N
        draws[done:done + m] = 1.0 - (counts @ sse) / sst[:, None]
        done += m
    return names, draws


def boot_p(draws):
    x = np.asarray(draws, float)
    lo = (1 + np.sum(x <= 0)) / (len(x) + 1)
    hi = (1 + np.sum(x >= 0)) / (len(x) + 1)
    return float(min(1.0, 2 * min(lo, hi)))


def fdr(p):
    p = np.asarray(p, float)
    n = len(p)
    order = np.argsort(p)
    adj = np.empty(n)
    adj[order] = np.minimum.accumulate(
        (p[order] * n / np.arange(1, n + 1))[::-1])[::-1]
    return np.clip(adj, 0, 1)


def r2(y, pred):
    return float(1 - np.sum((y - pred) ** 2) / np.sum((y - y.mean()) ** 2))


# ===========================================================================


def main():
    for language in LANGUAGES:
        for run in RUNS:
            out = OUT_ROOT / language / run
            coef_table = pd.read_csv(out / "trf_coefficients.csv")
            rng = np.random.default_rng(SEED)
            boots, contrasts, shapes = [], [], []
            t0 = time.time()

            for outcome in OUTCOMES:
                df = pd.read_csv(out / f"lagged_{outcome}.csv.gz")
                codes, uniq = pd.factorize(df["word"].astype(str), sort=True)
                n_words = len(uniq)

                for spec, lags in SPECS.items():
                    X_all, y, all_cols = prepare(df, lags)
                    where = {c: i for i, c in enumerate(all_cols)}

                    for feature, (model, feats) in CURVES.items():
                        cols = [f"{f}_lag{l}" for f in feats for l in lags]
                        X = X_all[:, [where[c] for c in cols]]

                        sel = coef_table[
                            (coef_table.spec == spec) &
                            (coef_table.outcome == outcome) &
                            (coef_table.model == model)]
                        alpha = float(sel["alpha"].iloc[0])

                        draws = bootstrap_coefficients(X, y, codes, n_words,
                                                       alpha, rng)
                        idx = [cols.index(f"{feature}_lag{l}") for l in lags]
                        np.save(out / f"boot_{spec}_{outcome}_{feature}.npy",
                                draws[:, idx].astype(np.float32))

                        beta = sel[sel.feature == feature].sort_values(
                            "lag_decades")["beta"].to_numpy()
                        for j, lag in zip(idx, lags):
                            d = draws[:, j]
                            boots.append(dict(
                                language=language, run=run, spec=spec,
                                outcome=outcome, model=model, feature=feature,
                                lag_decades=lag, lag_years=lag * STEP,
                                beta=float(draws[:, j].mean()),
                                boot_SE=float(d.std(ddof=1)),
                                CI_low=float(np.percentile(d, 2.5)),
                                CI_high=float(np.percentile(d, 97.5)),
                                p_boot=boot_p(d), alpha=alpha,
                                n_words=n_words, B=N_BOOT))

                        dc = draws[:, idx].sum(1)
                        w = np.abs(beta) / np.abs(beta).sum()
                        shapes.append(dict(
                            language=language, run=run, spec=spec,
                            outcome=outcome, model=model, feature=feature,
                            beta_first=float(beta[0]),
                            dc_gain=float(beta.sum()),
                            dc_CI_low=float(np.percentile(dc, 2.5)),
                            dc_CI_high=float(np.percentile(dc, 97.5)),
                            dc_p=boot_p(dc),
                            share_beyond_first=float(
                                np.abs(beta[1:]).sum() / np.abs(beta).sum()),
                            centre_of_gravity_years=float(
                                np.sum(w * np.array(lags)) * STEP)))

                    # ---- incremental R2 from the stored OOF predictions ----
                    oof = pd.read_csv(out / f"oof_{spec}_{outcome}.csv.gz")
                    obs = oof["observed"].to_numpy()
                    preds = {c[5:]: oof[c].to_numpy()
                             for c in oof.columns if c.startswith("pred_")}
                    names, draws = bootstrap_r2(obs, preds, codes, n_words,
                                                rng)
                    pos = {m: i for i, m in enumerate(names)}
                    for label, base, new in CONTRASTS:
                        d = draws[:, pos[new]] - draws[:, pos[base]]
                        contrasts.append(dict(
                            language=language, run=run, spec=spec,
                            outcome=outcome, contrast=label, base=base,
                            new=new,
                            delta_R2=r2(obs, preds[new]) - r2(obs, preds[base]),
                            CI_low=float(np.percentile(d, 2.5)),
                            CI_high=float(np.percentile(d, 97.5)),
                            p_boot=boot_p(d)))
                    print(f"[{language}/{run}] {outcome} {spec} done",
                          flush=True)

            boot_df = pd.DataFrame(boots)
            boot_df["p_fdr"] = fdr(boot_df["p_boot"].to_numpy())
            boot_df.to_csv(out / "trf_bootstrap.csv", index=False)
            pd.DataFrame(contrasts).to_csv(out / "model_contrasts.csv",
                                           index=False)
            pd.DataFrame(shapes).to_csv(out / "trf_shape_summary.csv",
                                        index=False)
            print(f"  -> {(time.time() - t0) / 60:.1f} min\n", flush=True)


if __name__ == "__main__":
    main()


[English/main] RSC main done
[English/main] RSC lag1plus done
[English/main] Turnover main done
[English/main] Turnover lag1plus done
  -> 0.1 min

[English/content] RSC main done
[English/content] RSC lag1plus done
[English/content] Turnover main done
[English/content] Turnover lag1plus done
  -> 0.1 min

[English/window] RSC main done
[English/window] RSC lag1plus done
[English/window] Turnover main done
[English/window] Turnover lag1plus done
  -> 0.1 min

[French/main] RSC main done
[French/main] RSC lag1plus done
[French/main] Turnover main done
[French/main] Turnover lag1plus done
  -> 0.1 min

[French/content] RSC main done
[French/content] RSC lag1plus done
[French/content] Turnover main done
[French/content] Turnover lag1plus done
  -> 0.1 min

[French/window] RSC main done
[French/window] RSC lag1plus done
[French/window] Turnover main done
[French/window] Turnover lag1plus done
  -> 0.1 min

